# Ronda 7: YOLO con la foto vista de tres formas

Idea de jhonatan (24 oct 2026): enseñar al detector la misma foto procesada de otras maneras para que contraste
patrones. YOLO recibe tres canales: **original**, **contraste local** (CLAHE) y **solo lo oscuro y fino**
(black-hat: quita el brillo general del disco). Todo lo demás igual que la ronda 5: YOLO11m-seg a 1024 px, 586
imágenes de estudio, mismo refinador de la ronda 3, mismo examen de 121.

1. Entrena YOLO (tope `HORAS`).
2. Examen: candidatos y nota por umbral, con aciertos, inventados y escapados. Compara con la ronda 5.
3. Test: `submission.csv` con el mejor umbral del examen. **No envía nada.**

Requisitos en Kaggle: GPU T4, **Internet On** y como *Input*: la competición y el notebook
`filamentos-detector-ronda-2` Version 4 (`refinador.pt`).

In [ ]:
# Configuración. PRUEBA_RAPIDA=True reduce todo para comprobar que el notebook funciona.
PRUEBA_RAPIDA = False

RES = 1024
MODELO = 'yolo11m-seg.pt'
HORAS = 2.5                # mismo tope que la ronda 5
EPOCAS = 150
PACIENCIA = 25
LOTE = 6
CONF_MIN = 0.1
UMBRAL_REF = 0.5
UMBRALES = (0.15, 0.2, 0.25, 0.3, 0.35, 0.4)
REF_R5 = dict(PQ=0.4560, umbral=0.25, aciertos=878, inventados=381, escapados=489)  # ronda 5, mismo examen

In [ ]:
import subprocess, sys
for paquete, modulo in (('ultralytics', 'ultralytics'), ('segmentation-models-pytorch', 'segmentation_models_pytorch'),
                        ('pycocotools', 'pycocotools')):
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paquete], check=True)

## Código del proyecto\nEmpaquetado del repositorio `filamentos-solares-2026` (los ficheros de `filamentos/`).

In [ ]:
import base64, io, os, tarfile
tarfile.open(fileobj=io.BytesIO(base64.b64decode('H4sIAAsK3WoC/+19W3fbRpZunvUraph1OoADwbyIks00s9ptO+nMOI6PnfSss9hcXBBYpCCBAAKADGmP8w/mR/RjHvIwq9/mVX/s7EtVoQCCkpJO55yeiCuRSaCuu6p27ctXu/yH/sM/vAq2f5LBXOYf/EM+Xf4c+rfbHQyq7/i81+33eh+I7Qe/wmddlEEO1X/w2/z0z8SqjFZy3Dt73B+cnAxPH/ndk+7grH/0wf3nf/5nEcXBSiZlWjyczaIkKmczP9v98uv/9OTk4PrvdQcf9Ib902G/2z07w/V/Auk/EN1fc/3naVrelO629/+kn06n8/T6b/NomYq5jEWWpzsZlvhDFHKJMyMIo+u/JfigmiuiSOMgl4UP2e/5xD/zx///Yv8/2d//+/f7/6+y/z+q7f+nj/o9/+xxbzg8Gd4v7N/W/j8P4O8vvvnfvv+fnPTN/j8cdE9h/Q9Oht37/f9X2v9fwIa/zgPc4b988vln0YuvxE6EabKReUFbP0yPXIosja9/XKZJWnhidf3XIgzyoICUr188JzEgWmVpXorLIk3097Q4WuTpCgqLY6gkSpNCqFdzuQjWcTmPwvJIJ0/Wq2wnoNAk43zZLkzDtEzT2ORbBcUVJsF/Z+syioujo6+ffCnGot89eXR0dAQFC2jaMshnQZKi9JImsnDyIHrrjo4EfKCxz+RmLeONFO+SdHWey1m0CpYyGYl3lGee5iMxUdmRAk+/evqVJ3zfn75/7x9RKd8kgeBcIltLoF0pE5mLiwDmkxiIXC6josyRWOskBeLlQhfti+exiOYkb+lkVKQsROdz+Dx5cvx7btennU/EughWIHHxCyDlKoXMMCTRIgqxOCrHlK17SP9+F5UXIs1k4uDCDsoL/zKNEqKFJzol/Jt04Isa9VnP786eJFgSDdXsKlguY9nv9k9nlNbHse24LtJ/wbTED44R0B9f+nEazJ2FS+903+Ddu2jSiead6Ug48A14jpwlwHY6U0+oN36RxVHpdI477qQ7dcUCOhaJKKHSIQlSuuhM31PJ6bqEQq0p5AATO58Ho/ozqN3lpmBp0JxZNPeEw6T1iGYuVqEb6kelXBWOW3UNKppw8ilNh6kQH8KoX/8Vhh1Gi6kO1CigFEs6LqE10fUP0Pg1CtCJaUNQ9SioCA10MTXajYM+6qZNAkWDGRJreriBfpDBeM+dQI8BLO1EvLsC0iBNNkzZK09ssCmQXXf6vVo7sMwjXOWzYAZrDBe5UykCaQIti8t0DEsO2xhe0FdFsTwGmoytpekv8le7r84vYe0XLaWoEmottTLPZZjOpWM9Wcl8KR2sxnV9WGi7TDrnwB1c1XbV4GIG2azFr5vHNUxaegjU1c3DEemoCUjDZRU0rdcDBUBbHPWr4i5PQZROiJ9+FKbrpCw+QiaJv5mPwHqF0lcBq1lxgPxxvYIcdhuwaF8vZPheJ6xMiDZJBmSAsmB5JkGeBzvdGE0dSLCOkvKR69aoDOVNOtw2WHqK0B3IGkUdd6+TmmCFLt0TZRpLqDOMgjFwDavrKSyAHJJiz9bQJb1TMOPCJQnLZbLtemIH/2978G+POatwNFGQSq4nMswGtNpEBTA2TQraGKChsFPobWNOK4M2jnlkJbri53rnkEGxziVz7x3w5S1OVsjhL6JkPkvVND1IQGBLlDVOwyDGnBmw1LlOP+ESgaH1mNKGaJB0MjUcIMQppZrCFYdIsnVeOFTwXr2e6PpDiyWFOBFUfljrebqFXpYSB2kH9TmhPTiuyRYtRAz7QOiKT8diUBVXa6pmHpMcpgbwceDmyDKwT9T4HXSROBj9Ig7ibMXH0G8fpahSHAscUHiys56409rcM7WpeQZz0VqJPClv5DKHGMW7ThG9lR3cuSvmAgOiJ/pI8Be9ePR8f9/gJff6/73+/1vV/5X9HwZgcHav//+29P8wDoooTH95C8At+v9pt9tr2P8HZyfDe/3/19L/r39MZCDOg0KKMAZ5DfTKkUjPC5lvSMF03qYJCFdpEa5BuHZhe79cy7fCAeUkxucqJbwC3fgNisxZDmJE9Da4BKlsXaYgBJY4s8SbKAelmwU7UGtASs4b0rBMNtc/pmSAWKVHuVxISAWSDBokQMpFafnbtQQJaB7lIIoVYgVbOfzrH6GASHLfPAB5HRpWyMraoB8dtjW8+uKFTv0Fio1HtwqaNwiZHjQrzy7SOF3uQMz5Q1U//RWvgAIrieYBlmnWoMEF8YwonI4ECV4g53X9R4/wNeicSSmTIpoHc5B/YlAQNgGISEC4ZJ6yRMqyUTCPUpCltnYZj7ta0PtQ0JCJ/kgkMF4knEMxMOZkRICE/4utElgMFRjkEmSzKBlhC6C0waBrS466wMEIdd7g+r9SKPPHJFpxbuoPNpqLqJp0cmLnPhlBF7gxiVzmqShSlO9AfAuMp4lbE6NhZxUhv6oX2fMfVeUNRzyjVPp5wDROUKvKtrorp0LUuwLD8RYmFJBFKgVGoi1njjoLTcECVRqJ9jD4t9L3lRwL6cJ0Rs11otXS0onQfoZTmshKBKa0UBOshDLKSQuEGrPrH7cSdejzPIrjAIa8MGrPCloMpYpPgXoD8YC0D5mHaAcCFRTeeOLxY9ckRbXmPILltZtBO+MZTEVZONZTtApFydJZeSIqQVsgQ8R4oNTEODj3xEyVAz9k7Kz4zRVrPkBXmGWopUB5JFY7kMzPgw0kBV2pN5oin+ixsoV6w5YzJmnyVuYpphbjsbiqqQbbLaj4QeKAxrHbma+Qq/g2hwpAf9luXfGQOh9pPTXBoY6B2eRMhnAL/0ONry1zX7SJ5pKMcLK2aISDjAR0ZxzTDdlqAhr573vdrsi2rqG/3QfQ2qIQyAnV+cVFkEnVCX57scvSEhoKqg82ZbfDLzts9mtKNufpMIb0v1cEgiUNnUs5/3cXwPIcTuUJ6hI85XZilRN+NVVDlUnkhzwoBdsAiH/5yJ74tyreUi1p2Qz6ruvD0gKlyXGGvb4n4A/Qm7P/8YsXX7x8/uS1saAh0XhCcFtwYsHMcagBnsBixv1hI/XhRlESU78h5WQ0Ou5NDzQCh1qtA5oEMAWj1XrFZRnNG1NNvldUQt5gTzF86YlczZ1qk3PUC09ktiEHuktukZGo7YF6EM1aRisvsCXotOIiqzVsYbBAA1Sr9SRi/g5Notpg+EHJtzm/K34Huyo9N5zcredsWcL8rraOFSU+FC/QGskt0q1BNhYs83UWoCkyISvNHHiZsnUDAWB94j5rTDcLqC9G6zQ8TPNoiT+pfCyFDBzaIMGswmojFQxNMo2EXX+N7gY5rnZIH0h45QAtFIduGKtULQ8UEdTQoQjiyDKCyVcGhacG1hq8J1kMgoxACUUJKiBE4K4X6lEFljzXjgB8X3k1glBmZNk1Q2ee1M05ULJt0EFzbZpkeZoVVdss4020wBw+7qo0ymZ7rdljUNSJkrU0D8lU1mVT2bbHZmH//DzdmhSbICZrFk79XXdECbujbW86waQkokztVmB6Hyp2XGpHfaO+uTG6D9uogPl5meYzYC7L8gKWJG4J1bsoMe/IgkQdbu7gN9e1Yh6CO4ZFUOYUnmBzjUlb77iiEnXdpKnGVVm6VrWZZl5rW3gu5zKMcidfl4HyEcEcG1fiG+x13AXkSS38jpwvVm7XZ8da6XRedIDXEiPmNph9C10bDVnC8DWPd5mDW57dG1oh7RyuWi73Sv49/ufe/ndv/yP733DQG56CdtZ/dHJ2OrhnDb8x/I8sEfqZ/8r430EfZh3b/86Gp70h4X97p/f4n1/L/vdMjTvo67lEWZ/gLyQqm9nB7tt1ItBNCj8LiUa8eeofHfV8YUpwvkR4zuvjpy9feig9EXIIC0T9mty6I5DCgb2fgHKNYOMsTeR+VQSYQL0kDC4Dj7+lyQKUzrcBCO34u4hilAV9NKWhCQWfkkVQoMQGGvz1DzFKOZSY5H//qO+L17qDwvnm+KUsWX2+/q8AGpbLMM3LoNkcXhaqD7EB/diKEFSH6CPok0cqFai4mCSIUX6EytAnCl+j8/UltHwNgmsOWQLRH55u4X8g4pP823VUKiBWlBRZlGMr0NgUQy9KCbLulciu//sclBoylL4ERTpJorD3WDidiuwwiNSxI+qpzDvuJ1hEqDDeoMSBfE1glyMLtGVbQ9UjGM7w4ibT6N3AWXe0omICv4A5VUg/LJZ5kF3o9EDQBAgj0VO+wvmSlHXM1+vnT796/fVz/Dk8Bcn9Q3H8d36EZoUKTgYqNaitUT7Tzx17bo+/ztEAkkvQu2FmK4WA+kVE3EQFKIU+GarRWV0yFM4mWh4myQwKSGQ57M4WWTLb9D2Bo/oaxnT2WhYv6c1nr17O/tyf/buMlhdlcbd6fPKD5zweutIn9OxzhKwF2M+7lbQIilLmM2ytLugzeIRtfIXzO7x7UTRF7IJ0Z28pCDRrk2Mdl9GbMIjl6/SLJ3G0TI6URawgo8Rt5POfPf/syTcvvkaNts6r4gLXVyKNOfXQGDnfcVFjqtMT6ufsPAivzqGAMZaibTHP0AgTB4UsRspAtrOMyFSVn6fR7AJE8cIH7X6WaVpAE/YI7RxM7YdxMQMFMpd+lMwWMkB7C7Su7+7VQsNgV7M3DE6lYB/MR0rtEDEcV1hjeAGcCYjo3TFLui6tPH1Nr6fA/NHSVgC3653ihrEzFv1COGgxtzB3KwRFiv7xqdjIULIxh+FVOfA9tJEiy63wZtC0zKyNpV4IQIHG0nDQPlmMHad36oHe7Az69M/pCf3T6z+if4HzeK7rHYmf8gmKDFbDjIx2WEPXH3qi53eBBn7X9VzxQAw1Md7wfodGrP6jLUjtZLB+i53qnWx7JyPd4TKIY3Sj2bSBnSAt2scef2bAw3Ho99aTg3NnFWQE1izGk04X4aI9/NPHPwMEcMLoZTCAbP995IkiWGVxlCy5X+OWSWe4QDHLgJ2w6aTX7aqECGZDUNsKDVRUrld/GmzpKdpRcV67xHtr2KA6DniZRwUZYShhHUZ02GJTN9UYQzmtJCxmz0ZNdnoGbTXxiJDeBiRSAR66zCxcE2zH39iAObRuSXRNTDAx/D8lB4YtmKBAViF/qZQ3MM9ZzghYJkF5xN6qHQbiYUU/lDlaSNNFFEZB7MLyAr4HC3QdSpTN2MyPMh4W/IxKJQeKQYJDVTnKRtgelGfOYcVi19J4zbhprBylB5ksQeQJyMtFwk8x0mKeZXWFybDA4oA00L5X/xvG7/pHwlF2vUePoXz4p8/etwAauJbnJBeK3rDC5Fa2dsIIW6baJpizMj3ejFvdw4bSwNEft2ao/MnQVcsR07R4HvKarO4yEV3yEg5ti+0KON3ORjYrErXbQeGFmsGzBUwbNHVGQcOpiRBn9pKStzL1lIceJASYSMtgtQrQd5vuuUqEs1hLAgCwt7fysRnPmHZ0dtl1A+1P0CMCc3FF/5DVuuufWYb1yi/G+c2vBw+szKAjIAsBTvsIOa12V92hAloM9aKVg/JAGxzbVyeOuQBXHGgNMf7eEH2mNyZ8TOncBgcL4yhjS3CXvGBHCmbwDF1XWjlzSIzy1QQNQHeC1yCNlDb/EfJSAvvmAwuOtnhrHgNKEo6i1vYY5iFpDRZ6f0qufyyjeaAAwEJuYVWXqacs2JQWmQ+RE1hBnhagxL2A9HTcBBemgQ5HxcqoWyj7g6ZXU8EkOl5WoIiBUqjc1rYTp9CsEDUxgX6eVEkT5Gw3x1mEs1wHwJ1yzF6m2HZsaHBO2+hrYNA0Q9UUhTWhzkk7hYwXnqaYZurBmlhFbq01TOdXyegnpaVvOgPMmQNF4f4/UYMhaIOp9gTPZmnu1G5lLBPVSKstatKgL73WLNfOuZQlngowXYzc2kqr1QnNrhU0iabNyshmEG3SmerQPOXJenh71CRyGzRyjd/25hL3hgEGEJmKIt6fvH+f4kozU0XvtYU4/lQ4CLXB4ydq9uvK6ICUUbRdVvWXEcxgNf3CJk+EVWEaApM8tWaxvY5Y+oyBbUp0s4JQCNsxok4MZ2QRCZ3k+1uHeAh67/DI2jwgZ3ilgemFS/xf/WDdxrjUnK6HLMd4322fGuTSba9GX8E/LFYJFHVO3Pr0UO3I0/JxlwfmiqUjfgKb5xXKzijr1xDqrUx4WN+0TAXIXkeeUHiBFX1XP+3WQMrGJgZyXQZ7oXvUbC/QCB2Qy3W6Vtsvbnle+yu1Y4akpFheSoKprCDXCees7fAoh1xCfZcoi8gEyAtCMsgJ1vqqTiZUYJnVZUUmqnByiVvMVrlwPYTbq2/4LNg6BL7h5/oXS0V6Lo/Fuw4ojRJh8bwxoJAxI0OQQ3VAYR3y51dJ6CSd7t8cJ+KYX8AkOD1pUX86KAG1VrHaP2Dx3t7V9jLgUExQnZ4CY8hAL3EGHnax51Yr1EADcNKCgljKurBfrrNYOm9hs3xAL5Gd/IFrSlJQEwKYc4zlYpsf6NiELlQwHBBfMk9j9dSKGlenM+o8xgjtyFHaOM3kpVcxcWPexOdTVx8oA3kqp63YvK8AYdQyX26CWMkgiE5qI9tBnnGAnn6ZOthVLpUkY6rLmWyn5hQMLhmnmKgBnuLq607xbEmdPKBEZWvH9bkxaslAX3DXALkabRR4AolPmrQlDkLEOlIjSIRFaMHYZh1a5ZuolIqY5qeaE+qo1sEzJIcPVLUdpeIyGZhCbvoMtqCiYrlmQlz/J2PFFOJmjkiiOW0RMMHUBoq2CWDKZRqCpOJ8kX6Dsq8rNiy3oBSUclVm7BMgBy1DXZ+RYoFj9vc2+wnsydWBSugZaCKJO7XP7E00fSjdijAtqnBOh62rqyYW8aJ0TaqNJ/gvTBKUX1kRoewzkJMUfANKaTPpOliDpzRKOR9/FsBOVRviCe6JPIuDUrPGqlQEE6rTjVU39euKDbrVjED92EwJRAyBAIMKJ8mz62ATvUVr4HkUR4kMYpJ92XIPRaIyGxWpZYbBTR0oFmsRti+ybXWA74C1wdLsboTn0UkotBUcVPa4Uyx3zBRGWElFlhD0R5ZL5srpwohTIPommkfz9GcgJJWZwVXmhy9ptobXf43DdcxmNzqdDA+iIqKMGtir4eMnIPcACwWq53m0pPQERwIpiHFC4iJA9K1kzBmPwI7UgHgNu56JT1PA9hzEvujRDFulpiefgFo0xCWDOgUIi0ZAQ4ht/cT0TYLW/1v05j8ZVBNafTtSExIdBGrCu+YY9E4157VhZPYKfn6J82sHs/ghKlOE5M7VIVsJSjKwWSAB7AsxWazYFqYh/BZ4z8L9toljht8SJLlAOXWwx3V7PrM+dVY0Rmfh0pfRErF3F9ipMN04RlyfbAuU16Z71iEQrOpbHW+WGhWNjC2e4Dj05PFjV6HwYmDB6onZrqJEAdmrjYr3SgKF8b4N4sypJxZr1MSDXHm2NEpx3PWqEwaQ8vFwT+KDUSOfh6fAmpSNH9CYUU5yilixGKqtMKjLTaQEEc+okJls1SxZlyGMm2I9utEjMgO0bLC8TycNGD/ZqCqjquXBZvlDuRO0r1rqHdmyw6YFJ4JO0w8FuGUntMsMJh9ZBwmArYbMR2TCFMWO2YZTjK1lAX+RjIphgl4xItMGbjborgYds1iLbI1NIR+AwgqTMZYPMRh9VO8LZtgcXqeNbcP9BB3paJ27/rGIcImsZYw0zNkwQxycT2b41cEMHFvdyOF+I7VPRrlw1OZQIM0vIrTI4PM1V1i4dbZ8JWXGcsOikjFZDtWK7dh8nWDqqV6iZlpUprwqywQXZrqMQoRg5z7bvs3qmCxVHADC+B6W9rgu1KyrStBjk5jKEcOgN1uacBwyBE1joPuHEq3T9pGSkVjSlhvryV8pAFXoh5AESlv3zNuRsXG0yiIbuaxV0ZqcV1ezyaAtfie+V9XUDceq6v+ANHXl/ICl2WpFw+LcGBCVaN+CT6JV0EetA13/D4XqKwULefBA9I9u6RQ1rYAtCE2rurTfG94mEMw9WWmjLghV1VmqW/HQ1iKN0HhYkvcYxJ65Yta40WHZqKPjblFluL3wam01y27sgtBmk/aWYi3LvxGDWzwAvwCSwiCKtEfuMpiF62CO8BY0BDFUZtxDgzyQA9b++PFpi2vsKedJmXEqNA4KeHrJ6KIUvwaWGGMqlgYcpIDcAq8xmJw7bfEMrMZJp20tHO2g0iZQk/aEs6te75qvlWmalipay3Dnl1FMO7cp61hYNdiWm2Nhld3Dycv9NCe1qFxMQPt+gOIVE9K1fFQccEM1gKM7hHhCibI/RGXas1/taq/qJeiq8EGX61CpXc+83O2/rB300vE/utApbvNOf9VhIRiNpRHuBATDU5B5Nm4InCOrfTqkCG4WkOWOuhfK5bXDA5WwrcBFnlBfXN0KI1ZxQ2UxYwAJ6Vy4pCprNP/at0i/woONuGNSAcjqtcnZeqRFH7fCi1UhpczsVzLQU9BcdT0kvJG8wuuCrMtK7NgEsLMHJP9kKZ5eoiO5SlszSxbdIHOJXCJQuo9OYkzXMqHFLGtL0ThyyS982CNb49RIJHuPpjaPW5xgPURJ9P2+a1uo2a4NXKRmJ1WzoIXh1MzO+2Zua2VhAZM+evDoW3daSzQHBjHftRrFj3kBPRS9rieq77y8LY9v6+JSdUFirAFXEu56e8uMkvUo2W4vWX0LYHI4N608t21/cOyFyDMZy3K9aoWuWoLW8ILlhYar9Pmbr922LcZyVL5WC+k2R2WLD47XDUrgDROccb9tlSdpR1ppMzXKa8pu+9LDFQb//3Rf2vbOTjQ8F6odZ9tJND1ootWtbk9zd7eMqtC4YLYNl8zO00dxf4IrRhW6tdwwu8meD0YdmiUf9fZmtzbMw/23x12/e4JRkLroUeka44E1As6e2bvVZbN1D3kQ9oTVuxW40wUedCQwL80dw1PJJtS2PdjOhTEjAtBHMR706Wwps/Nq73it4cR1tDKGWNjj4RtJx/1xLmnTpQFQMy5DI/+P9En5nO2v1flIWzsxpbNrAhuwTlT0BpTNAtzsEqV1Z1Ehk8qRuSARtp3pazuvskzrLtg+Du1umzT4utu+oeiSwsL2jE5aWJpbReeiOqbuza7WLE/PG2dBLes6cnyEkUHFLo+jvfjb/DSYdBKNImTFkHzq4kKqTVaYqe0eGlY3XzFgi4bhBGOmIRhjk8alZLjk118/qZKjMt2tqYLzaFVQPC+HQI4MalR/xcBt6HPbCNc90AXWNOakvZiKoH14Ww8vFpkOF9FylUbzajk428jV3p/6hoVNzIAaThYdrAeDAJj0OCBm08pgqE5c9krVvUpubfjICpeEAajh6Iilx+4BacWzdk8SMjFGHdIMHYr1xUxTyONaLNKBCrYJbra1ZpXoue2hVkDHflEL6Lba/mvGEGuFUVV3A30d1BI/hAlYMMfACKQKukMABvxBpyJYaiTv0AGmMmJ+xAIt2T2sCqIkjNdRbpnMLNgqaHIIXKUaEo4KgsYsk906/V2PVUGdt7azuRpqYGyQoTpIPGmeGf6dqF6SF27qNgmL6xoK/PlUtSZBVIAcZ/UCy7XCqq6ymP1lFbjA8gPVbTw69d4haKquTbDTOVybMSPr32PIKKlpmxYZOK0x0iu5ULw+SDxjYHXIIcM6Qw3xUnfrtdrgJhjZ1HZb3myLg9Tu9P7A4j/r+d/e3vnf7tn9+d9f5fzv0Dr/e9Z7/Gjw2O/3Tx/fr6XfXvw/E+3mFz4BfPP5397pyYDj/58OB8OzwRme/z0569+f//2Vzv++CopU9EhmK3NQKhOFdbGMfIgASxmnmhYpoy8WEToBNZLETB7QIZ9bedn3nqfJHKPiay9qmRK+hESDDQPIqwO+n7Kjj32eFb6m6z1y8agx1XdUBfEBaYPx22jz1wKoE9eFUG7y3NaMPa25BqTG6hPNHmHvUA46IvGCQMiYyPQXpPwlYn49cr2i1DOX5OAzQq4KXE1REp/4ZJWVBbU0uMT5lpCLOpOEQSmObMoLPhwB1fwbRd/n5nz+6hvhzNOweBhYB4WPdSZ/NXcPnuf9xx/h5RS+ObDLcUQbT6tTA4V4ps/vPvPh36PPXj95+a9P3qB5suv30NTTxz8D/EOWnyH+OcU/Z/jnEf2hh4/pDx+b6xmLPHru6sdVnrX43qsTKeTJ9ipnvLKHWxApY8Cwo/TogKmNaD1tQK47x5BTBq2b48i1niDSqDbyc1VIR6KyRvfVADa3RoCfS1UYp6tjJ6to21onuUuobXOsCBcTmuNzGcMfpTUTFmU1qw7p3eCa2+l4WF0VE4vcTtpVdiwQJNV16xDogSe0062RyvbpDRjwiBopFlvXqBjukBCaohaNzUQ6o4z1aGxdDOz2/d0S7+HMZNyI6EV5lF2CmEVbIm6iwurK+UynNOAhBUmj567LSAn46tPJSuVN6doDTvdH4BCN7ejw+ECB3dGVPka7s/KzoxdgW0uMVCb3uttudQ13teS7enIVCR5rUKYRV8WJx0fGSLJXKq3pWsE03ypnv2p/WoQzoEotZUUoHhCVdq8OzJsNm1mtCJ0aETjUtfEAqRxmhNpLNyfrZoTJvKWFCgJmldmTx4MDDSfEgCrwmV9HFMz4BM7AuDph+waeBVxOOSlhtS7RWi+1+VrHQPDMDj4mjq5A4ZBjTNwcvU0ztV0iqkzj7RvQMOuOHYX7QjB9JacaRynZ2vBkubmr0BY61PUTaI0Lco1U1VWOYN4nnA3dwAdg+zeC9akC9AmIlPcVFTyCAFYEqVWH6v7PVy++cj8hcQPIiYnoMhAregd0VbUvtGJPGhwcrkFthyMW6omPgLN/BMKBNq2ApIOPkNYficrQqmFoil6jyt2F+AV2BOLfqT71h6eBWZhyRcqYExB36gPXxM5agUNuCopXWTt59vzUw9V6X6v2eX5EM+RmLO2dENUWgPWZ30RU6wmfM+z5gFOObl5imzcGwA4hmcNLRCbBeQxcBn8haL0TrudBx3ZXBjGMO91vpA6hYA/J8K0eMHt+5psEtBzNERUdMmCdVEIX483mcpPGG0SbzSWhytbM1pUfR0vUTubDlPVJSLXmuhb1N2o2xsL4FGAjx3tJuOmTUV9htrYl7TmE21fv+lNXA2f5CcKW+tyjybv3ZOKbkRdmakKYKsQlKgvWsnZIvAatwIReVfK6HccIkd/6KhnDApTJGsFE5EDi8R7QcNsgWehUHRr76AZo7N897qoBlsew4q75oOEgrBiqWwNI2sqTGRnPjMSKsZHwTH+ZSCI51HGlXRmU2KMiaayulBShltcExUnCjG3ZIktXwqymdjsmERtu1X0x1kk3WjVY6padE1OrlSsWA29oLKWomYVNNzDT/pu/e1B4IpHP8cDYrFr8t9XweJY31TWMnXcu9h96NScTkCyE/+lUIqokW+NgsklkmqWoZbV4QQceWbxeNuRrLNhymS38dTbHUZFb6+GkQ5MaquhMFWBslSuJzk4FmwzGQuBvJrnRP/DgpPmR2+cXofvaB7GoX1BkbS6jNvTKM7/h9l22wWjaMGBG3LnLfnQIC2a3j6Kz9A/sHfpzfkswhwbqbA9Gg1UMh6YBe8LbDS0ymJujOjw/NLSv3OGw2M+nZttjAcH2l5Mbe79xFAg4VK7YqnH24WItW5g21n7Zm3pz/KsgTDWltVo8SuzLB1pZXVO8Z3QGqUSgnm/k2MxOi49bJzYJJEfbA0LQK6GL9ohcZhEe1laH8jg/AvGUnNeAZjdRDtSOPW+aoU/X9uHhqabExNYvoHfO5HiLKnqCC8vwWSoTlt1VlMzHHRgf4F8dt3aUr1LbMS3wxAlRwr5UkWpTEPl0/RPP9RGygQKZKFV1nc2gQC6lzKO1g2VqchFeAVl916S2PZp0KRm2s+bRhEQTbiPKQVx+TRddZ2rYkQDBbAHr9jJQIz8DaWZWISkDaGrtUWNeWOf/wghjVTCykkoUHN3CRB4M19d/VRGVLA8o1sBucjqxGcTQjFBacrxd/0i8018mZrpN33+y31I75URdIFhF1Jiq+0hZBn9aD2yON7lQqAE+zEhim8rXMHY2owH5FDK+vg4U5XktHJr85uQqCFwUP4OvviRLsjFuagBSro9ZrhMdXkgDjRwkbWGNgR5D2MHLgCejoTjQenn934nVcyL+Wy7cNMjTk0pxCHMuZb530kTNpPqSUAZKfSKZZknRnMUqJ1mVXHVmttQ3Bd6UDLbWYGaW4U0pMf5cwKtc9fOcfyUzXlZdI0jgPMeZZ3iGNbH27znl3AdYaLWFcDUfs1AAX5tSAdcILGjLDAukIi2+qAIhk6vZ43zajMBUsa5tjWdZnM8CPNApcuKXOEaTg8xyDyZJUETc3WQAMiHyWjnX1NZWyCKaI4/AcH4dl+81VKAua1gsBCZfBkCDQzfKNtbyJJnWJRozlB+PFVQMf7nN0zRIJyTOqNUmVMOVmNj8QWv6O/J5GTQ5PTesIUiwWfGSAuZBbjKe9twW61I1S9W0cXTyrjq57DZn0f5hqmaNtWXCW1GduNDL+dwPSoeWIIUb6R1KotczpeIauF0H0uvlSunNiV8royoFDcvqiC1xtENYwb2lXjuzRB2APbxODCpRC3JkmOUSxovOO1UW5vH7i/fH79DWbR6igZsuKaWXHc9i9WRG1fXdGstQk41y6R+UUWRhOVNPaqZKPDH+QFhpxUNR1ddqEd6bwzMCOdaK1QPCBeLUsqrAYtmGWT9DiwT0zCr07JnqMZ/7bSAg7uP/38f/N/ifR2fdR8OBf9Z73Dsd9O8hQL8t/M8ujf8Bl3/eiv/pD7qw2BH/c9bvnpx1T/H+z0H/Pv7/r4X/eU0G9eGo8hagn6rXOy7kUjjfYBCqeFdGIYgnGH5xvQzyeig8vPWTz35IjtTPF3Oi0b4QlTVFHoQFOTUFlWO3KJdZQHBzz7gBj9TlAejnyvKUYtVa2ipWXoVoUNeKqsqzfZyT8gzkEuVgvE8Jo8xYCCE8rh+t1jG5EghCtEnVPWkal1PD3aTVBaPhpv8PRuP8pHj5DMaxKGVgOaxxqQS1d4jNUVaWHLQZPlNSx9Xw3DkTDqhLZC25vEiToEQUOAxTGpai3+2fuiOOvYPxPDEWn3EhUcE8ToVnoFjQyiD2KxeiHbXRwWDOLnpnq6fewIjHoFRYl9R9op72XCs4rQpr9PTFkz89p1sfgvhtQBdHXP9wHsU0rjBjVhzrWGGFTLigCjWkC++7okMxHeA/FXpjRxG3O8I5j4Pw6vgiKDG+EduOajE6ZFGLw0RHpta4ktIcTSh8Y58lFSvIHELACGGmwnroAJIVoglvtrtUl98FO+viQrb31C1YDkb6oKtWz3MpoM1JmCqzCN0fyDe7Fnj4ArogQOkoyIpCa0df5uFUx8PK6x8zvKXie7wRA0d5ma9lkaoA0xx7IdFubGWSqBlhIImKwqRjPIiHVBFb1OLgAmMmwQrzQxi+UtJYOqjWv4hWUTnuYwx1hF18nkfzNxjtCKO3i94pxl7OsnhXOY2vamfi+0MODuG64j9UOCJUJXJdXXVn4fOtOpgOT7/86vWrP82evvjqzXN+sJTlG3XPYZQsn8cS6eJUSZ+/ePHFK0zsXOG5S7dxxyMmLNbnMF8RcUP1k3u3kaw/HIK+qO0Z/Ljt0NoDcUrGC3IlNK34jRPyyiPAXUM660A+oEgd9yocCNkP8xmxC0dhMTCCK5Ad2OYGOSu7d21MCDMQ5QuzIknnGUI2MWS9Yti4/4yIHRj0B/IJvDLGxGU1i5yufiE0hGS7ZHWCJ1MgDL2zOHEjqnChbfw0G8mwmaSrc6S3TOLgLZ1lxHYgBpWQHFySyyGLYz5lp040qQ1QBS3bIRWAAGqzQAZBh512MlGOdYrJYhVK19XsXzBjvApoN1C7ITdSHdjrwESJkg6SHs/rdaDeDo2BfWovRYDbFd6aXTiLzjsenPcPqa7i4TsqF80Bcgt9mqVXfNz0tuwcH/T27GSRxNaqhtetGXTz+BjryILyAra1OMI9yUnc5nF7pAm5SQ524eE7LO29nyXLTpsxTVdCDS0cLrAlWg1DV6wrAaBGmufvHxK5+e9MV5687/zk+wL261yoJTJqtYPYTaqciPbWbFn3kF202OyIpwYbqTtetyoxECBIkkLHyp8k031rsbGJ0WDyaBDVZ+9gEuwlm8O+EVF8vBtGjctqGbfDY6fKddupBYmL3SqOkitHN48K93R79olDOAE92O1TvGpouS3xCozvOi4KaIuL9lZYFw8kB4ZVH3a1Bm/vygGK5sZ3hcJOiHLZQRMZY2syOhp6OrrRoLa48L/LYXSdTld0xMfwf8e/TCPq/mbkDxfvO9SBDa1ctc9k6gw+QmU7f0k6Fu6lSTo+S7ILVvEBSpkGLDo4sCPxTg9xcF7gvxoj9P4vCa03BYWb0Q8cgr8kGzytwE/hKz+j60pGf0lAHhxVgo5prGGkxD8N+yTGWY90TtHNlX+Asi3Ie1r4WCEwwxtYqcuBobg7UaEn4qH5D7S2lmv7TKROqhy3zL9qaKHXelD9tlqrbmF/66OqjaWtQ2pA4up6AR0q+RAGtH33/8wCX94SOtnCYJKreA+IqbyZJpqBxscpVUI1AHcm8kpyg0kkpZ37+m8kBdgMFfZ5S/+FlcV3B7x6+bmOdlBt+Jj3Kxiyp38Wzh8/f+2qoPCQG4QbPgmL17XgAmcoG+IOCjxZDSy3uP5RhymMkg1arzkQKqECCHiXkLfIxOkJYUhDGBLr8oxFHTHYCDjQ0OAmvu+rSB3VvmOiw1diILqBBiz7GWediQjtq+ucDoXbIKm1eDumPZCg43pK4DQBJhzQBVA8Jjca//HOH+jdeNAF5gOb7HlaSDWVbBkB/V90r1GBwdIItEohMPjYdu5TxPPGhtGEZSAuw9q0CZTg1VAbh+5tUVVzzJpa0IHGhSwhJabGoNCwqCe+KdBLC0zkOLRawTjMlQJPIHTOgnrcKZ72oZjaVUBtG4ehJhzwgj9/8eZrPsPj0Lh4gocHJFKGSpqf6jVJiOYt/XIR7+lw0AqQ89+CnivzeXA8l7kksDNMrOg8OA7Og8vUbTCgWVkGd2VC3FhLB7H5BkbAyFCXWEUUisaSyXU8DVySmyBHAZ6NFxxfA4R+ij3PmRAaUmR422JQ6PhcFX4WD47R6sEjXuvmvSUKDYGaD4ZcSIuIuaSl9HCUU4oqCmwTg33wGmY2wsgCFfHUhmbgITSKRa4DoFYn98SbtYXnkEV1UQuZ60wNTlcoJYYjMxwp0wTyrTQhNX9t+uPxbStk4FCgDZBcQxXcXCZpUYvLHNc7Ykr1DUKEuSpetXH9A97GbkOX6eKoqgt1xLLzERf8kULsKzBwSrFkVTBqbdTTXFWpJm2bXDW1rLBPBzjwClPD/5sK8uk1g3FuYC3EiDaeu3XMLLe6GYGlcQMGCX4KGQ7P99JuJybZet4WpUWxDuzwwehJ+xFC7Vs37CasDiRttqGRjuEQCBdvRtGo3oT1Nxt6M9lovEK4F7FjVfyCDL82bBq2cpd7BLYt9whY8GkL2bIpbFzLxgKFoE0ipHvvfiZeTyEbMM7JpjARhsS/jKHSCYdpHll4GYrafyhePzemPWK/iup6pRZJUYG6D0z+Wsh+K35m4+6XuTo9Wb8BgGP/16MHwX7Aq76BFlGLqX0Tr6EzoLJ99cmUO9kgumCqzh02H+NqR1BDfaamBjWGiJfJqqjh1aE+vM0J9+NuY6WRVKHQHlzKA9FHBY+D0TRXJpHcQKwJI2EaqI+8NeA2ODo1PInaKCe143t0hqyatKbQqXuzhGImM7VsX9CwsMjU1xpdWIBR1/Aw1IuKaUlUX6l7g9dySEAVfkCmoaOqsCXMMARXGe0B40ZHNupNKYoqxZXcjUHxPEdc5kgchxqddgirWwO8hQ3AWysSzIRLM6vJYGsJhft38YgmbFdXFnL7bkP2Qbk5Y+KslFxhAwLIVC5AOKBw7bo4DR7eA3HXA9BXIty/rjlMgHErDtW5PGNFHtVjDOga9IF7dbkHH7g3A+7pe8Xr97Q0QKtxUJ3RQ6QuOhBj6zLKUJEbz2bcMJ+OVLhKnBm5JnRuCK0KIeqZ+VR1hWJCw3OiELz4/bgi1tS+sQBqwFHuVVPp1ttdCjxGQn+q+YLFWFe83LphHL7gxer23pWZE3iB113R1VnKENXG/qfujRtJTW2BItW8w0vnZky+u4HH7zgvSS6doecsSZsmD7zYlN3Ul805OxCBguCK5oGEA7P3wOysT09zbDT7FkYR/i8zmPf4f6LCWFStZXkdnXQZO/mAqMoDSIvEjgJpOSdK5DwL+pMYIDIi8/CHOkjOFRQU58HDC3/h74D+9roYw/2xa0K9c93W3j2guMvuz8Q38+S6mclYQ/jzYb2KrxVku2GefQCPu8g09BfX0f7r5CZkMGvykhHse4BYztCMEHcDrPb2pf9TcMEXUUmXB9RtHm3NhpT2HlMLwGpw5iZDG07YzLKPdWiD1QRKnR5Ky2QlHPL3pmD3YOraGIljyqgb3WvNhnfJm7VEHNki1j7p8ahd4+5hfHRw0Ku18TGN2QXsOukyD1YOlTSpOsXXFON6MxYyFsyQsjgyMI4L62tiRb0kCTD7dkyUfUj56PhxwkpbF93rhXkNBSJQObNeltmYWcyYucx4kZgwgxV5RnZYwknH9G1GZoFZ70pdL26/6l3NBm3PgYD0gmiZOTBQXkWsA5cbq+WMoVHQ16XWdMWo3ynH7EhM1O2k139Lpu/RLG0O5NTD705tf+6URQk08vIlfsbE07wWbLK/7d39MmpzkN2te5ume2qN7djTrHITxGtZOPfBE+8/95/7z/3n/nP/uf/cf+4/95/7z/3n/nP/+R/8+b9Xy/keAMgAAA==')), mode='r:gz').extractall('.')
print(sorted(os.listdir('filamentos')))

## Datos, refinador y separación estudio / examen\nMisma separación que las rondas 1-6.

In [ ]:
import csv, glob, hashlib, json, shutil, time
import numpy as np, pandas as pd, torch
import segmentation_models_pytorch as smp
from PIL import Image
from ultralytics import YOLO
from filamentos.datos import cargar_anotaciones
from filamentos import candidatos as C, yolo as Y

disp = 'cuda' if torch.cuda.is_available() else 'cpu'
T0 = time.time()

def buscar(patron):
    for raiz in ('/kaggle/input', os.environ.get('MAGFILO', '.'), os.environ.get('PESOS', '.')):
        r = sorted(glob.glob(f'{raiz}/**/{patron}', recursive=True))
        if r:
            return r[0]
    return None

json_path = buscar('MAGFiLO_1.0_Annotations_kaggle2026_train.json')
assert json_path, 'No encuentro los datos: añade la competición como Input del notebook'
DATOS = os.path.dirname(os.path.dirname(json_path))
PESOS_REF = buscar('refinador.pt')
assert PESOS_REF, ('No encuentro refinador.pt: añade como Input el notebook filamentos-detector-ronda-2 (Version 4). '
                   'Hay en /kaggle/input: ' + str(glob.glob('/kaggle/input/*/*')))
print('Datos en', DATOS, '| refinador:', PESOS_REF)

ANOT = cargar_anotaciones(DATOS)
nombres = sorted(n for n in ANOT if os.path.exists(f'{DATOS}/train/train_images/{n}'))

def es_val(n):
    return int(hashlib.md5(n.encode()).hexdigest(), 16) % 100 < 15
val = [n for n in nombres if es_val(n)]
tr = [n for n in nombres if not es_val(n)]
TEST = os.environ.get('TEST_DIR', f'{DATOS}/test/test_images')
test = sorted(os.listdir(TEST)) if os.path.isdir(TEST) else []
if PRUEBA_RAPIDA:
    tr, val, test = tr[:6], val[:3], test[:2]
    RES, MODELO, EPOCAS, HORAS, LOTE = 256, 'yolo11n-seg.pt', 1, 0.05, 4
print(len(tr), 'imágenes de estudio,', len(val), 'de examen,', len(test), 'de test')

ref = smp.Unet('resnet18', encoder_weights=None, in_channels=3, classes=1).to(disp)
ref.load_state_dict(torch.load(PESOS_REF, map_location=disp))
reales_val = {n: Y.reales_rle(ANOT[n]) for n in val}

## Las tres vistas de una imagen\nPara comprobar a ojo lo que ve YOLO: original, contraste local, solo lo oscuro y fino.

In [ ]:
muestra = np.asarray(Image.open(f'{DATOS}/train/train_images/{val[0]}').convert('L').resize((RES, RES), Image.BILINEAR))
v = Y.tres_vistas(muestra)
Image.fromarray(np.concatenate([v[..., i] for i in range(3)], 1)).save('tres_vistas.png')
from IPython.display import Image as Ver, display
display(Ver('tres_vistas.png', width=1200))

## 1. Entrenar YOLO con las tres vistas\nMismos ajustes que la ronda 5.

In [ ]:
t = time.time()
yaml = Y.preparar_datos(DATOS, ANOT, tr, val, '/tmp/yolo_datos', res=RES, vistas=True)
print(f'Datos YOLO listos en {time.time() - t:.0f} s')
modelo = YOLO(MODELO)
modelo.train(data=yaml, imgsz=RES, epochs=EPOCAS, time=HORAS, patience=PACIENCIA, batch=LOTE, single_cls=True,
             fliplr=0.5, flipud=0.5, degrees=0, hsv_h=0, hsv_s=0, hsv_v=0.3, mosaic=1.0, close_mosaic=10,
             overlap_mask=False, cos_lr=True, project='/tmp/yolo_runs', name='r7', exist_ok=True,
             plots=False, workers=2, device=0 if disp == 'cuda' else 'cpu', verbose=False)
shutil.copy('/tmp/yolo_runs/r7/weights/best.pt', 'yolo_tres_vistas.pt')
shutil.copy('/tmp/yolo_runs/r7/results.csv', 'yolo_historial.csv')
print(pd.read_csv('yolo_historial.csv').iloc[:, :12].tail(8).to_string(index=False))
print(f'Entrenamiento: {(time.time() - t) / 60:.0f} min')

## 2. Examen\nYOLO en modo generoso + refinador de la ronda 3; se guardan todos los candidatos para analizar sin GPU.

In [ ]:
det = Y.detector(YOLO('yolo_tres_vistas.pt'), RES, CONF_MIN, vistas=True)
t = time.time()
cand_val = {}
for i, n in enumerate(val):
    grande = np.asarray(Image.open(f'{DATOS}/train/train_images/{n}').convert('L'), np.uint8)
    cand_val[n], _, _ = C.procesar(None, ref, grande, disp, RES, CONF_MIN, UMBRAL_REF, 0, detectar=det)
    if i % 20 == 0:
        print(f'{i + 1}/{len(val)} imágenes, {time.time() - t:.0f} s', flush=True)
with open('candidatos_tres_vistas_examen.json', 'w') as fh:
    json.dump(cand_val, fh, separators=(',', ':'))

pd.set_option('display.width', 200)
for clave, nombre in (('rle', 'YOLO solo'), ('rle_ref', 'YOLO + refinador')):
    filas = []
    for u in UMBRALES:
        r = Y.nota_umbral(cand_val, reales_val, u, clave, por_tamano=True)
        filas.append(dict(umbral=u, PQ=round(r['pq'], 4), SQ=round(r['sq'], 3), aciertos=r['tp'], inventados=r['fp'],
                          escapados=r['fn'], esc_menos_1k=r['escapados_menos_1k'], esc_1k_3k=r['escapados_1k_3k'],
                          esc_mas_3k=r['escapados_mas_3k']))
    tabla = pd.DataFrame(filas)
    print(f'\n=== Examen, {nombre}, tres vistas ===')
    print(tabla.to_string(index=False))
mejor = tabla.loc[tabla.PQ.idxmax()]
UMBRAL = float(mejor.umbral)
print(f"\nRonda 5 (una vista): PQ {REF_R5['PQ']} umbral {REF_R5['umbral']} | aciertos {REF_R5['aciertos']} "
      f"inventados {REF_R5['inventados']} escapados {REF_R5['escapados']}")
print(f'Ronda 7 (tres vistas): PQ {mejor.PQ} umbral {UMBRAL} | aciertos {int(mejor.aciertos)} '
      f'inventados {int(mejor.inventados)} escapados {int(mejor.escapados)}')

## 3. Test y `submission.csv`\nCon el mejor umbral del examen. El archivo se genera pero **no se envía**.

In [ ]:
t = time.time()
filas, cand_test = [], {}
for i, f in enumerate(test):
    grande = np.asarray(Image.open(f'{TEST}/{f}').convert('L'), np.uint8)
    cand_test[f], _, _ = C.procesar(None, ref, grande, disp, RES, CONF_MIN, UMBRAL_REF, 0, detectar=det)
    img_id = os.path.splitext(f)[0]
    for k, r in enumerate(Y.seleccion(cand_test[f], UMBRAL), start=1):
        filas.append((f'{img_id}_{k}', r['counts'].decode('ascii') if isinstance(r['counts'], bytes) else r['counts']))
    if i % 30 == 0:
        print(f'{i + 1}/{len(test)} imágenes, {len(filas)} filamentos, {time.time() - t:.0f} s', flush=True)
with open('candidatos_test.json', 'w') as fh:
    json.dump(cand_test, fh, separators=(',', ':'))
with open('submission.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['filament_id', 'segmentation_rle'])
    w.writerows(filas)

ids = [a for a, _ in filas]
assert len(ids) == len(set(ids))
if filas:
    from filamentos.datos import rle_a_mascara
    assert rle_a_mascara(filas[0][1]).shape == (2048, 2048)
else:
    print('AVISO: no hay filamentos en el test (¿falta la carpeta test?)')
print(len(filas), 'filamentos en', len(set(i.rsplit('_', 1)[0] for i in ids)), 'imágenes -> submission.csv')
print(f'Tiempo total: {(time.time() - T0) / 60:.0f} min')